In [1]:
import numpy as np
from scipy.stats import norm

# Paramètres du trade : call spread 80/90 sur le TTF Nov26
F = 74.28 # Prix du future (EUR/MWh)
K1 = 80 # Strike du call acheté
K2 = 90 # Strike du call vendu
sigma = 0.75 # Volatilité implicite
jours = 36 # Jours calendaires jusqu'à l'échéance des options
T = jours / 365 # Maturité en années
r = 0.02 # Taux sans risque, sert juste à actualiser
volume = 100_000 # Volume en MWh pour avoir des montants en euros


# Modèle de Black 76 : comme Black-Scholes mais sur un future (pas de r dans d1)
def black76(F, K, sigma, T, r):
    d1 = (np.log(F / K) + sigma**2 / 2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    actu = np.exp(-r * T) # Facteur d'actualisation

    call = actu * (F * norm.cdf(d1) - K * norm.cdf(d2))
    delta = actu * norm.cdf(d1)
    vega = actu * F * norm.pdf(d1) * np.sqrt(T) / 100 # Pour +1 point de vol
    theta = (- actu * F * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) + r * call) / 365 # Par jour
    proba = norm.cdf(d2) # Proba risque-neutre de finir au-dessus du strike

    return {"call": call, "delta": delta, "vega": vega, "theta": theta, "proba": proba}


# Prix des deux jambes et du spread
jambe1 = black76(F, K1, sigma, T, r) # Call 80 acheté
jambe2 = black76(F, K2, sigma, T, r) # Call 90 vendu
prime = jambe1["call"] - jambe2["call"] # Ce qu'on paye pour le spread (= perte max)
largeur = K2 - K1
gain_max = largeur - prime # Si F finit au-dessus de 90
point_mort = K1 + prime
proba_zone = prime / (largeur * np.exp(-r * T)) # Prime / largeur = proba approx de finir dans la zone

# Greeks du spread : jambe achetée - jambe vendue
delta = jambe1["delta"] - jambe2["delta"]
vega = jambe1["vega"] - jambe2["vega"]
theta = jambe1["theta"] - jambe2["theta"]


# Affichage des resultats
print(f"\nCall spread {K1}/{K2} sur TTF Nov26 (F = {F:.2f}, vol = {sigma:.0%}, T = {jours} j)\n")
print(f"  Call {K1} achete : {jambe1['call']:6.3f} EUR/MWh")
print(f"  Call {K2} vendu : {jambe2['call']:6.3f} EUR/MWh")
print(f"  Prime nette : {prime:6.3f} EUR/MWh")
print(f"  Point mort : {point_mort:6.2f} ({point_mort / F - 1:+.1%} vs F)")
print(f"  Gain max / perte max : {gain_max:.2f} / {prime:.2f}  (ratio {gain_max / prime:.2f})")
print(f"  Proba implicite ~ : {proba_zone:.1%}")
print(f"  P(F_T > {K1}) / P(F_T > {K2}) : {jambe1['proba']:.1%} / {jambe2['proba']:.1%}")
print(f"  Delta {delta:.3f} | Vega {vega:.3f} /pt vol | Theta {theta:.3f} /jour")
print(f"\n  Pour {volume:,.0f} MWh : prime {prime * volume:,.0f} EUR, gain max {gain_max * volume:,.0f} EUR\n")


# Sensibilité de la prime à la vol : on reprice le spread avec plusieurs vols
print("  Sensibilite a la volatilite :")
for vol in [0.45, 0.60, 0.75, 0.90, 1.05, 1.20]:
    c1 = black76(F, K1, vol, T, r)["call"]
    c2 = black76(F, K2, vol, T, r)["call"]
    print(f"    vol {vol * 100:4.0f} %  ->  prime {c1 - c2:5.3f}  (call {K1} {c1:5.2f}, call {K2} {c2:5.2f})")


Call spread 80/90 sur TTF Nov26 (F = 74.28, vol = 75%, T = 36 j)

  Call 80 achete :  4.717 EUR/MWh
  Call 90 vendu :  2.241 EUR/MWh
  Prime nette :  2.476 EUR/MWh
  Point mort :  82.48 (+11.0% vs F)
  Gain max / perte max : 7.52 / 2.48  (ratio 3.04)
  Proba implicite ~ : 24.8%
  P(F_T > 80) / P(F_T > 90) : 33.3% / 17.5%
  Delta 0.179 | Vega 0.018 /pt vol | Theta -0.019 /jour

  Pour 100,000 MWh : prime 247,578 EUR, gain max 752,422 EUR

  Sensibilite a la volatilite :
    vol   45 %  ->  prime 1.603  (call 80  2.07, call 90  0.46)
    vol   60 %  ->  prime 2.133  (call 80  3.37, call 90  1.23)
    vol   75 %  ->  prime 2.476  (call 80  4.72, call 90  2.24)
    vol   90 %  ->  prime 2.698  (call 80  6.09, call 90  3.39)
    vol  105 %  ->  prime 2.841  (call 80  7.48, call 90  4.64)
    vol  120 %  ->  prime 2.932  (call 80  8.87, call 90  5.94)
